# 01 — Main Analysis
## Correlation-Based Leakage Screening Is Insufficient for IoT/IIoT Intrusion-Detection Benchmarks

Reproduces **Tables 2, 3, 5, 6** and **Figures 1–5** of the manuscript.

Fatma Mohammed Dhaou · University of Tabuk · fdhaou@ut.edu.sa · ORCID 0009-0000-9028-8033

---

**Data.** Obtain `ML-EdgeIIoT-dataset.csv` from Ferrag et al. (2022), *IEEE Access* 10:40281–40306, doi:10.1109/ACCESS.2022.3165809. Not redistributed here.

**Reproducibility note.** Exact duplicates must be removed **after** the thirteen identifier/payload fields are dropped. While `frame.time`, `ip.src_host`, `tcp.payload` etc. are present, nearly every record is unique and only ~814 duplicates appear. Once dropped, records identical in every *behavioural* feature collapse together and **5,555** duplicates are found, yielding **152,245** records. Cell 3 asserts this.

## 1 — Setup

In [ ]:
# In Colab, uncomment:
# !pip install -q shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CSV_PATH = "ML-EdgeIIoT-dataset.csv"   # adjust if needed

# Colab users: uncomment to upload
# from google.colab import files
# up = files.upload(); CSV_PATH = list(up.keys())[0]

print("Setup complete.")

## 2 — Load

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Expected: 157,800 rows x 63 columns")
print(f"\nAttack_type distribution:\n{df['Attack_type'].value_counts()}")

## 3 — Preprocessing (Section III-B)

Thirteen fields dropped, in two groups:
- **Raw payload / free text** — `http.file_data`, `http.request.uri.query`, `http.request.full_uri`, `http.referer`, `mqtt.msg`, `tcp.payload`, `tcp.options`. Risk encoding testbed artefacts (literal attack-tool strings, fixed target URLs).
- **Connection-level identifiers** — `ip.src_host`, `ip.dst_host`, `tcp.srcport`, `arp.*.proto_ipv4`, `frame.time`. Specific to the testbed's fixed addressing and capture timeline; retaining them risks learning *which machine* sent a flow rather than *whether it is malicious*.

**Order matters** — drop first, deduplicate second.

In [ ]:
DROP_13 = [
    "frame.time", "ip.src_host", "ip.dst_host",
    "arp.dst.proto_ipv4", "arp.src.proto_ipv4",
    "http.file_data", "http.request.uri.query", "http.request.full_uri",
    "http.referer", "tcp.options", "tcp.payload", "tcp.srcport", "mqtt.msg",
]

df = df.drop(columns=[c for c in DROP_13 if c in df.columns])
print(f"After dropping 13 fields: {df.shape[1]} columns")

n_before = len(df)
df = df.dropna().drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df):,} duplicate/null rows  (expected 5,555)")
print(f"Remaining: {len(df):,} rows")

assert len(df) == 152245, f"Expected 152,245 rows, got {len(df):,}. Check the drop-then-dedupe ordering."
print("\nCHECKPOINT PASSED — matches the manuscript.")

In [ ]:
# Label-encode the six low-cardinality categorical fields
CATEGORICAL = ["http.request.method", "http.request.version", "dns.qry.name.len",
               "mqtt.conack.flags", "mqtt.protoname", "mqtt.topic"]

for c in CATEGORICAL:
    if c in df.columns:
        df[c] = df[c].astype("category").cat.codes

y_bin   = df["Attack_label"].astype(int)
y_multi = df["Attack_type"]
X_all   = df.drop(columns=["Attack_label", "Attack_type"])

print(f"Predictive features: {X_all.shape[1]}   (expected 48)")
assert X_all.shape[1] == 48

## 4 — Leakage screen (Section III-C)

Absolute Pearson correlation between each feature and `Attack_label`. Threshold |r| > 0.85.

The threshold is a qualitative judgement, not a formal criterion: **no single legitimate network-behavioural feature should linearly separate fourteen structurally different attack types from benign traffic across a population of this size.**

This screen is univariate and linear. It can only detect univariate, linear shortcuts. Section IV-C of the manuscript shows that this distinction is not academic.

In [ ]:
corr = X_all.corrwith(y_bin).abs().sort_values(ascending=False)
print("Top 10 |correlation| with Attack_label:\n")
print(corr.head(10).to_string(float_format=lambda v: f"{v:.3f}"))

LEAKAGE_FIELDS = corr[corr > 0.85].index.tolist()
print(f"\nFields exceeding |r| > 0.85 -> {LEAKAGE_FIELDS}")
print("Manuscript reports: dns.qry.name.len (0.966), mqtt.topic (0.881),")
print("                    mqtt.protoname (0.880), mqtt.conack.flags (0.877)")

FEATURE_SETS = {
    "Uncontrolled (48)":       list(X_all.columns),
    "Leakage-controlled (44)": [c for c in X_all.columns if c not in LEAKAGE_FIELDS],
}
for k, v in FEATURE_SETS.items():
    print(f"  {k}: {len(v)} features")

## 5 — Binary classification (Tables 2 and 3)

Four classifiers spanning deliberately different inductive biases. Logistic Regression is not a competitive detector here — it is an **instrument for detecting linearly accessible shortcuts**.

Stratified 80/20 split on the 15-class label to preserve rare-class proportions (~40:1 imbalance).

In [ ]:
def build_models():
    return {
        "Random Forest":        RandomForestClassifier(n_estimators=200, max_depth=20,
                                                       random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting":    GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                                           random_state=RANDOM_STATE),
        "Logistic Regression":  LogisticRegression(penalty="l2", max_iter=2000,
                                                   random_state=RANDOM_STATE),
        "MLP (Neural Network)": MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                                              random_state=RANDOM_STATE),
    }

NEEDS_SCALING = {"Logistic Regression", "MLP (Neural Network)"}

def evaluate(feature_set_name, cols):
    X = X_all[cols]
    Xtr, Xte, ytr, yte = train_test_split(X, y_bin, test_size=0.20,
                                          random_state=RANDOM_STATE, stratify=y_multi)
    scaler = StandardScaler().fit(Xtr)          # fitted on TRAIN only
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

    rows, fitted = [], {}
    for name, model in build_models().items():
        a, b = (Xtr_s, Xte_s) if name in NEEDS_SCALING else (Xtr, Xte)
        model.fit(a, ytr)
        pred = model.predict(b)
        prob = model.predict_proba(b)[:, 1]
        rows.append({
            "Model": name,
            "Accuracy":  accuracy_score(yte, pred),
            "Precision": precision_score(yte, pred),
            "Recall":    recall_score(yte, pred),
            "F1-Score":  f1_score(yte, pred),
            "AUC":       roc_auc_score(yte, prob),
        })
        fitted[name] = (model, prob, pred)
        print(f"  {name:<22} acc={rows[-1]['Accuracy']:.4f}  AUC={rows[-1]['AUC']:.4f}")

    return pd.DataFrame(rows), fitted, (Xtr, Xte, ytr, yte)

print("TABLE 2 — Uncontrolled feature set (48 features)")
print("-" * 60)
tbl2, _, _ = evaluate("Uncontrolled", FEATURE_SETS["Uncontrolled (48)"])

print("\nTABLE 3 — Leakage-controlled feature set (44 features)")
print("-" * 60)
tbl3, fitted_ctrl, (Xtr_c, Xte_c, ytr_c, yte_c) = evaluate(
    "Controlled", FEATURE_SETS["Leakage-controlled (44)"])

In [ ]:
comp = tbl2[["Model", "Accuracy"]].merge(
    tbl3[["Model", "Accuracy"]], on="Model", suffixes=(" (48)", " (44)"))
comp["Delta"] = comp["Accuracy (44)"] - comp["Accuracy (48)"]

print("=" * 66)
print("THE CENTRAL RESULT")
print("=" * 66)
print(comp.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print("""
Only the LINEAR model degrades. Random Forest still classifies every one of
the 30,449 held-out records correctly after the four most strongly
label-correlated fields in the dataset have been removed.

Two readings are available:
  1. Genuine non-linear signal was there all along.
  2. The shortcut is REDUNDANTLY ENCODED — the screen removed the route a
     hyperplane could take and left others open.

The manuscript argues for (2), because zero error on a heterogeneous
14-class detection problem is not a property genuine problems have — and
because the SAME model on the SAME features finds application-layer attacks
genuinely difficult at the 15-class level (Section 7 below).

Neither reading can be settled without capture-session identifiers, which
the released CSV does not provide. See README, 'What this work does not claim'.
""")

## 6 — Figures 1 and 2

In [ ]:
# Figure 1 — ROC curves, leakage-controlled
plt.figure(figsize=(7, 5.5))
for name, (model, prob, pred) in fitted_ctrl.items():
    fpr, tpr, _ = roc_curve(yte_c, prob)
    plt.plot(fpr, tpr, lw=1.6, label=f"{name} (AUC={roc_auc_score(yte_c, prob):.4f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Binary IoT Attack Detection\n(Edge-IIoTset Test Set, Leakage-Controlled Features)")
plt.legend(loc="lower right", fontsize=8); plt.tight_layout()
plt.savefig("fig1_roc_curves.png", dpi=300); plt.show()

# Figure 2 — Random Forest confusion matrix
rf_pred = fitted_ctrl["Random Forest"][2]
cm = confusion_matrix(yte_c, rf_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest (Binary,\nLeakage-Controlled)")
plt.tight_layout(); plt.savefig("fig2_confusion_binary.png", dpi=300); plt.show()

print(f"Total errors: {(rf_pred != yte_c).sum()} of {len(yte_c):,} test records")

## 7 — Multi-class, 15 categories (Table 5, Figure 3)

**This is where the binary result becomes hard to defend as genuine signal.** The same Random Forest, on the same leakage-controlled features, cannot reliably separate `DDoS_HTTP` from `Normal` (precision ≈ 0.74) or `Password` from legitimate activity (precision ≈ 0.77).

A model that finds application-layer attacks genuinely difficult at the 15-class level, yet makes **zero errors** on the binary task that subsumes them, is a model whose binary task has been made trivial by something other than attack behaviour.

The network-layer vs. application-layer asymmetry is the manuscript's most robust substantive finding, because it is a **relative** comparison within one model — residual shortcut structure would inflate all fifteen classes, and cannot explain why one group separates perfectly and another does not.

In [ ]:
cols_c = FEATURE_SETS["Leakage-controlled (44)"]
Xtr_m, Xte_m, ytr_m, yte_m = train_test_split(
    X_all[cols_c], y_multi, test_size=0.20, random_state=RANDOM_STATE, stratify=y_multi)

rf_multi = RandomForestClassifier(n_estimators=200, max_depth=20,
                                  random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr_m, ytr_m)
pred_m = rf_multi.predict(Xte_m)

print(f"Overall accuracy: {accuracy_score(yte_m, pred_m):.3f}   (manuscript: 0.949)")
print(f"Macro-F1:         {f1_score(yte_m, pred_m, average='macro'):.3f}   (manuscript: 0.947)\n")
print(classification_report(yte_m, pred_m, digits=3))

labels = sorted(y_multi.unique())
plt.figure(figsize=(9, 7.5))
sns.heatmap(confusion_matrix(yte_m, pred_m, labels=labels), annot=True, fmt="d",
            cmap="Blues", cbar=False, xticklabels=labels, yticklabels=labels,
            annot_kws={"size": 7})
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest, 15-Class, Leakage-Controlled")
plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.savefig("fig3_confusion_multiclass.png", dpi=300); plt.show()

## 8 — Feature importance and SHAP (Table 6, Figures 4 and 5)

SHAP is used here **diagnostically**, not promotionally: to establish what a classifier that makes no errors is actually relying upon.

It answers *what the model uses*. It cannot answer *why that feature is predictive* — whether because of attack behaviour, or because of capture provenance. `tcp.dstport` is exactly the kind of field a per-scenario capture design would render session-diagnostic. Distinguishing the two requires session metadata the released CSV does not provide.

**Explainability is a necessary instrument for auditing an IDS. It is not a sufficient one.**

In [ ]:
import shap

rf_bin = fitted_ctrl["Random Forest"][0]

# Figure 4 — Gini importance
imp = pd.Series(rf_bin.feature_importances_, index=cols_c).sort_values(ascending=False)
plt.figure(figsize=(7.5, 5.5))
imp.head(15).iloc[::-1].plot(kind="barh", color="#2c5f8a")
plt.xlabel("Gini Importance")
plt.title("Top 15 Features — Random Forest (Leakage-Controlled)")
plt.tight_layout(); plt.savefig("fig4_feature_importance.png", dpi=300); plt.show()

# Figure 5 — SHAP summary (1,000-record stratified sample)
sample = Xte_c.sample(n=1000, random_state=RANDOM_STATE)
sv = shap.TreeExplainer(rf_bin).shap_values(sample)
sv_pos = sv[:, :, 1] if isinstance(sv, np.ndarray) and sv.ndim == 3 else (sv[1] if isinstance(sv, list) else sv)

shap.summary_plot(sv_pos, sample, show=False, max_display=15)
plt.title("SHAP Summary — Random Forest Binary (Leakage-Controlled)")
plt.tight_layout(); plt.savefig("fig5_shap_summary.png", dpi=300, bbox_inches="tight"); plt.show()

# Table 6 — Gini vs mean |SHAP|
tbl6 = pd.DataFrame({
    "RF Importance": imp,
    "Mean |SHAP|":   pd.Series(np.abs(sv_pos).mean(axis=0), index=cols_c),
}).sort_values("RF Importance", ascending=False).head(10)
tbl6.index.name = "Feature"

print("\nTABLE 6 — Top 10 features, Gini vs SHAP")
print(tbl6.to_string(float_format=lambda v: f"{v:.3f}"))
print("""
Both methods converge on TCP-layer features. Two readings:

  Benign      — port-targeting IS the behaviour of port scanning and several
                DDoS variants; TCP handshake irregularity is a meaningful
                indicator of malicious flows.
  Adversarial — if each attack scenario ran against a fixed service on a fixed
                port, then tcp.dstport encodes WHICH CAPTURE SESSION a record
                came from, just as mqtt.topic did — but non-linearly, and so
                invisibly to the correlation screen.

These are observationally equivalent from within the released dataset.
That indistinguishability IS the practical lesson of this study.
""")

---

## Next steps

The decisive experiments this notebook does **not** perform, in priority order:

1. **Grouped, session-aware cross-validation.** Reconstruct capture-session identifiers from the per-attack CSV files that precede the merged ML-ready release, assign group labels, and split by group. This is the test that would settle the question.
2. **Near-duplicate screening** beyond exact-duplicate removal.
3. **Ablation cascade.** Iteratively remove the top-*k* RF features and re-evaluate. If accuracy holds near 1.000 until most of the feature set is stripped, that is direct evidence of redundant shortcut encoding.
4. **Cross-dataset generalisation.** Train on Edge-IIoTset, evaluate on mapped classes in CICIoT2023 or N-BaIoT.

Contributions welcome — open an issue.